# Encoder training

In [ ]:
from bot import EncoderBoard, Plankton
from dataset import ChessBoardDataset

#from IPython.display import clear_output?
from torch.optim import Adam

In [ ]:
num_param = lambda m: sum(p.numel() for p in m.parameters())

def train(model, opt, data, loops = 64):
    model.train()
    hist_loss = []

    for _ in range(loops):
        loss = 0
        
        while True:
            data.move_white()
            if data.board.is_checkmate(): break

            move = model(data.board_to_ten(), data.move_white_history(10))
            loss = (loss + data.move_black_loss(move)) / 2
            if data.board.is_checkmate(): break

            clear_output()
            print(f"Game - {_}; Loss - {loss}")
            print(data.board)
        
        result = data.result_game()
        loss.backward()
        opt.step()
        model.zero_grad()
        hist_loss.append(loss)


In [ ]:
plankton = Plankton()
opt = Adam(
    plankton.parameters(),
    lr=1e-4,
)
data = ChessData()

print(f"Number parameters in model: {num_param(plankton)}")
print(plankton)

In [ ]:
train(
    model = plankton,
    opt = opt,
    data = data,
    loops = 100,
)

# PlanktonAI training

In [ ]:
from torch.profiler import profile, ProfilerActivity

input = torch.rand(1, 1, 32)
plankton.cpu().eval()

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    plankton(input)

print(prof.key_averages().table(sort_by="cpu_time_total"))

In [ ]:
#Save
input = torch.rand(5, 1, 32)
plankton.cpu().eval()
onnx_plankton = onnx.dynamo_export(plankton, input)
onnx_plankton.save("plankton.onnx")
print("succses")